# Tutorial - Modeling

This is a basic tutorial on how to use **_ROSS (Rotordynamics Open-Source Software)_**, a Python library for rotordynamic analysis. Most of this code follows object-oriented paradigm, which is represented in this 
[UML DIAGRAM](https://user-images.githubusercontent.com/32821252/50386686-131c5200-06d3-11e9-9806-f5746295be81.png).

Basically, we can divide the use of ROSS in two steps:
1. Building the model;

2. Simulating the system response.

This first part of the tutorial shows how to create materials, shafts, disks, bearings, seals, and point masses. Assembling those elements into a `Rotor` — plotting the model, adding nodes or elements, saving and loading, and concatenating rotors — is covered in [Tutorial - Assembling a Rotor](tutorial_modeling_part_2_rotor.ipynb). Then, if you need to learn how to run some analyses go to [Tutorial - Static and Modal Analyses](tutorial_analyses_part_1.ipynb) and [Tutorial - Time and Frequency Analyses](tutorial_analyses_part_2.ipynb).

In [1]:
from pathlib import Path
import ross as rs

# Make sure the default renderer is set to 'notebook_connected' for inline plots in Jupyter
import plotly.io as pio

pio.renderers.default = "notebook_connected"

/home/raphaelts/ross-wt-release-3.2.0/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Section 1: Material Class


There is a class called Material to hold material's properties. Materials are applied to shaft and disk elements.

## 1.1 Creating a material

To instantiate a `Material` class, you only need to give 2 out of the following parameters: `E` (Young's Modulus), `G_s` (Shear Modulus), `Poisson` (Poisson Coefficient), and the material density `rho`.

In [2]:
# from E and G_s
steel = rs.Material(name="Steel", rho=7810, E=211e9, G_s=81.2e9)
# from E and Poisson
steel2 = rs.Material(name="Steel", rho=7810, E=211e9, Poisson=0.3)
# from G_s and Poisson
steel3 = rs.Material(name="Steel", rho=7810, G_s=81.2e9, Poisson=0.3)

print(steel)

# returning attributes
print("=" * 36)
print(f"Young's Modulus: {steel.E}")
print(f"Shear Modulus:    {steel.G_s}")

Steel
-----------------------------------
Density         (kg/m**3): 7810.0
Young`s modulus (N/m**2):  2.11e+11
Shear modulus   (N/m**2):  8.12e+10
Poisson coefficient     :  0.29926108
Specific heat   (J/(kg*K)): None
Thermal conductivity (W/(m*K)): None
Young's Modulus: 211000000000.0
Shear Modulus:    81200000000.0


>**Note**: Adding 3 arguments to the Material class raises an error.

## 1.2 Saving materials

To save an already instantiated `Material` object, use the `save_material()` method. It writes the material, under its `name`, to the `available_materials.toml` file that ships with ROSS. Saving a material with the name of a packaged one, such as `"Steel"`, replaces that entry, so give your material a name of its own:

In [3]:
my_steel = rs.Material(name="my_steel", rho=7850, E=205e9, G_s=79.5e9)
my_steel.save_material()

## 1.3 Available materials

Saved Materials are stored in a **_.toml_** file, which can be read as **_.txt_**. The file is `available_materials.toml`, in the folder where ROSS is installed.

It's possible to access the Material data from the file. With the file opened, you can:

 - modify the properties directly;

 - create new materials.

It's important to **keep the file structure** to ensure the correct functioning of the class.

```
[Materials.Steel]
name = "Steel"
rho = 7810.0
E = 211000000000.0
G_s = 81200000000.0
Poisson = 0.2992610837438423
specific_heat = 434.0
thermal_conductivity = 60.5
color = "#525252"
```

⚠️ **Do not change the dictionary keys and the order they're built**.

To check what materials are available, use the command:

In [4]:
rs.Material.available_materials()

['Steel', 'AISI4140', 'A216WCB', 'brass', 'my_steel']

## 1.4 Loading materials

After checking the available materials, you should use the `Material.load_material('name')` method with the **name of the material** as a parameter.

In [5]:
my_steel = rs.Material.load_material("my_steel")
print(my_steel)

my_steel
-----------------------------------
Density         (kg/m**3): 7850.0
Young`s modulus (N/m**2):  2.05e+11
Shear modulus   (N/m**2):  7.95e+10
Poisson coefficient     :  0.28930818
Specific heat   (J/(kg*K)): None
Thermal conductivity (W/(m*K)): None


## 1.5 Removing materials

`Material.remove_material('name')` deletes a saved material from the file:

In [6]:
rs.Material.remove_material("my_steel")
rs.Material.available_materials()

['Steel', 'AISI4140', 'A216WCB', 'brass']

# Section 2: ShaftElement Class

`ShaftElement` allows you to create cylindrical and conical shaft elements. It means you can set different outer and inner diameters for each element node.

There are some ways in which you can choose the parameters to model this element:

- Euler–Bernoulli beam Theory (`rotary_inertia=False, shear_effects=False`)

- Timoshenko beam Theory (`rotary_inertia=True, shear_effects=True` — used as default)

The matrices (mass, stiffness, damping and gyroscopic) will be defined considering the following local coordinate vector:

$[x_0, y_0, \alpha_0, \beta_0, x_1, y_1, \alpha_1, \beta_1]^T$

where 
$\alpha_0$ and $\alpha_1$ are the bending on the yz plane
$\beta_0$ and $\beta_1$ are the bending on the xz plane.


This element represents the rotor's shaft, all the other elements are correlated with this one.

## 2.1 Creating shaft elements

The next examples present different ways of how to create a `ShaftElement` object, from a single element to a list of several shaft elements with different properties.

When creating shaft elements, you don't necessarily need to input a specific node. If `n=None`, the `Rotor` class will assign a value to the element when building a rotor model.

You can also pass the same `n` value to several shaft elements in the same rotor model. 

### 2.1.1 Solid cylindrical shaft element

A cylindrical shaft element with solid cross-section is defined when the internal diameters are zero and the external diameters are equal along the element length:

- `idl` == `idr` == 0 

- `odl` == `odr`

In this case, the geometry is constant and fully solid. To simplify the definition, you may provide only idl and odl, and the right-side diameters (`idr`, `odr`) will automatically assume the same values.

>**Note**: you can give all the 4 parameters, as long as they match each other.

In [7]:
# Cylindrical shaft element
L = 0.25
i_d = 0
o_d = 0.05
cy_elem = rs.ShaftElement(
    L=L,
    idl=i_d,
    odl=o_d,
    material=steel,
    shear_effects=True,
    rotary_inertia=True,
    gyroscopic=True,
)

print(cy_elem)

Element Number:             None
Element Length   (m):       0.25
Left Int. Diam.  (m):        0.0
Left Out. Diam.  (m):       0.05
Right Int. Diam. (m):        0.0
Right Out. Diam. (m):       0.05
-----------------------------------
Steel
-----------------------------------
Density         (kg/m**3): 7810.0
Young`s modulus (N/m**2):  2.11e+11
Shear modulus   (N/m**2):  8.12e+10
Poisson coefficient     :  0.29926108
Specific heat   (J/(kg*K)): None
Thermal conductivity (W/(m*K)): None


This creates a solid cylindrical element with a constant outer diameter, as shown below:

In [8]:
rs.Rotor(shaft_elements=[cy_elem]).plot_rotor().show()

### 2.1.2 Solid conical shaft element

A solid conical shaft element is defined when the internal diameters remain zero, but the external diameters vary along the element:

- `idl` == `idr` == 0

- `odl` != `odr`

In this case, all four diameter parameters must be explicitly provided, since the geometry changes from left to right.

In [9]:
# Conical shaft element
L = 0.25
idl = 0
idr = 0
odl = 0.05
odr = 0.07
co_elem = rs.ShaftElement(
    L=L,
    idl=idl,
    idr=idr,
    odl=odl,
    odr=odr,
    material=steel,
    shear_effects=True,
    rotary_inertia=True,
    gyroscopic=True,
)
print(co_elem)

Element Number:             None
Element Length   (m):       0.25
Left Int. Diam.  (m):        0.0
Left Out. Diam.  (m):       0.05
Right Int. Diam. (m):        0.0
Right Out. Diam. (m):       0.07
-----------------------------------
Steel
-----------------------------------
Density         (kg/m**3): 7810.0
Young`s modulus (N/m**2):  2.11e+11
Shear modulus   (N/m**2):  8.12e+10
Poisson coefficient     :  0.29926108
Specific heat   (J/(kg*K)): None
Thermal conductivity (W/(m*K)): None


This defines a solid conical shaft with an outer diameter that increases along the length of the element, as shown below:

In [10]:
rs.Rotor(shaft_elements=[co_elem]).plot_rotor().show()

### 2.1.3 Hollow cylindrical shaft element

A hollow cylindrical shaft element is defined when the internal diameters are non-zero and equal, and the external diameters are also equal:

- `idl` == `idr` != 0

- `odl` == `odr` 

This represents a tube with constant wall thickness along its length.

As in the solid cylindrical case, you may define only `idl` and `odl`, and the right-side diameters will be automatically assigned.

In [11]:
# Cylindrical shaft element (Hollow)
L = 0.25
i_d = 0.01
o_d = 0.05
ch_elem = rs.ShaftElement(
    L=L,
    idl=i_d,
    odl=o_d,
    material=steel,
    shear_effects=True,
    rotary_inertia=True,
    gyroscopic=True,
)

print(ch_elem)

Element Number:             None
Element Length   (m):       0.25
Left Int. Diam.  (m):       0.01
Left Out. Diam.  (m):       0.05
Right Int. Diam. (m):       0.01
Right Out. Diam. (m):       0.05
-----------------------------------
Steel
-----------------------------------
Density         (kg/m**3): 7810.0
Young`s modulus (N/m**2):  2.11e+11
Shear modulus   (N/m**2):  8.12e+10
Poisson coefficient     :  0.29926108
Specific heat   (J/(kg*K)): None
Thermal conductivity (W/(m*K)): None


This creates a hollow cylindrical element with constant inner and outer diameters.

In [12]:
rs.Rotor(shaft_elements=[ch_elem]).plot_rotor().show()

### 2.1.4 Hollow conical shaft element

A hollow conical shaft element is defined when the internal diameters are non-zero and equal, but the external diameters vary along the element:

- `idl` == `idr` != 0

- `odl` != `odr`

This represents a hollow truncated cone. All four diameter parameters must be provided explicitly.

In [13]:
# Conical shaft element (Hollow - Truncaterd Cone)
L = 0.25
idl = 0.02
idr = 0.02
odl = 0.05
odr = 0.07
coh_elem = rs.ShaftElement(
    L=L,
    idl=idl,
    idr=idr,
    odl=odl,
    odr=odr,
    material=steel,
    shear_effects=True,
    rotary_inertia=True,
    gyroscopic=True,
)
print(coh_elem)

Element Number:             None
Element Length   (m):       0.25
Left Int. Diam.  (m):       0.02
Left Out. Diam.  (m):       0.05
Right Int. Diam. (m):       0.02
Right Out. Diam. (m):       0.07
-----------------------------------
Steel
-----------------------------------
Density         (kg/m**3): 7810.0
Young`s modulus (N/m**2):  2.11e+11
Shear modulus   (N/m**2):  8.12e+10
Poisson coefficient     :  0.29926108
Specific heat   (J/(kg*K)): None
Thermal conductivity (W/(m*K)): None


In [14]:
rs.Rotor(shaft_elements=[coh_elem]).plot_rotor().show()

#### Returning element matrices

Use one of these methods to return the matrices:

- `.M()`: returns the mass matrix

- `.K(frequency)`: returns the stiffness matrix

- `.C(frequency)`: returns the damping matrix

- `.G()`: returns the gyroscopic matrix

In [15]:
# Mass matrix
# cy_elem.M()

# Stiffness matrix
# frequency = 0
# cy_elem.K(frequency)

# Damping matrix
# frequency = 0
# cy_elem.C(frequency)

# Gyroscopic matrix
# cy_elem.G()

### 2.1.5 List of elements - identical properties

Now we learned how to create elements, let's automate the process of creating multiple elements with identical properties.

In this example, we want 6 shaft elements with identical properties. This process can be done using a `for` loop or a list comprehension.

In [16]:
# Creating a list of shaft elements
L = 0.25
i_d = 0
o_d = 0.05
N = 6
shaft_elements = [
    rs.ShaftElement(
        L=L,
        idl=i_d,
        odl=o_d,
        material=steel,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for _ in range(N)
]
shaft_elements

[ShaftElement(L=0.25, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.25, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.25, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.25, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.25, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.25, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None)]

### 2.1.6 List of elements - different properties

Now we learned how to create elements, let's automate the process of creating multiple elements with different properties.

In this example, we want 6 shaft elements which properties may not be the same. This process can be done using a `for` loop or a list comprehension, coupled with Python's `zip()` method.

We create lists for each property, where each term refers to a single element:

In [17]:
# OPTION No.1:
# Using zip() method
L = [0.20, 0.20, 0.10, 0.10, 0.20, 0.20]
i_d = [0.01, 0, 0, 0, 0, 0.01]
o_d = [0.05, 0.05, 0.06, 0.06, 0.05, 0.05]
shaft_elements = [
    rs.ShaftElement(
        L=l,
        idl=idl,
        odl=odl,
        material=steel,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for l, idl, odl in zip(L, i_d, o_d, strict=True)
]
shaft_elements

[ShaftElement(L=0.2, idl=0.01, idr=0.01, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.1, idl=0.0, idr=0.0, odl=0.06,  odr=0.06, material='Steel', n=None),
 ShaftElement(L=0.1, idl=0.0, idr=0.0, odl=0.06,  odr=0.06, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.01, idr=0.01, odl=0.05,  odr=0.05, material='Steel', n=None)]

In [18]:
# OPTION No.2:
# Using list index
L = [0.20, 0.20, 0.10, 0.10, 0.20, 0.20]
i_d = [0.01, 0, 0, 0, 0, 0.01]
o_d = [0.05, 0.05, 0.06, 0.06, 0.05, 0.05]
N = len(L)
shaft_elements = [
    rs.ShaftElement(
        L=L[i],
        idl=i_d[i],
        odl=o_d[i],
        material=steel,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(N)
]
shaft_elements

[ShaftElement(L=0.2, idl=0.01, idr=0.01, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.1, idl=0.0, idr=0.0, odl=0.06,  odr=0.06, material='Steel', n=None),
 ShaftElement(L=0.1, idl=0.0, idr=0.0, odl=0.06,  odr=0.06, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.0, idr=0.0, odl=0.05,  odr=0.05, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.01, idr=0.01, odl=0.05,  odr=0.05, material='Steel', n=None)]

## 2.2 Creating shaft elements via Excel

There is an option for creating a list of shaft elements via an Excel file. The classmethod `.from_table()` reads an Excel file created and converts it to a list of shaft elements.

A header with the names of the columns is required. These names should match the names expected by the routine (usually the names of the parameters, but also similar ones). The program will read every row below the header until they end or it reaches a **_NaN_**, which means if the code reaches to an empty line, it stops iterating.

An example of Excel content can be found at ROSS GitHub repository at *ross/data/shaft_si.xls*, spreadsheet **_Model_**.

You can load it using the following code.

```python
shaft_file = Path("shaft_si.xls")
shaft = rs.ShaftElement.from_table(
    file=shaft_file, sheet_type="Model", sheet_name="Model"
)
```

## 2.3 Creating coupling element (CouplingElement Class)

Here, we introduce the `CouplingElement` class, a subclass of the `ShaftElement` class, designed to model the interaction between two rotor shafts. This element is implemented in a general form in ROSS. The coupling plays a crucial role in mechanical systems by transmitting forces, vibrations, and motion between rotating shafts. It is primarily characterized by adding stiffness, mass, and inertia to the system, which are essential for simulating the real-world behavior of coupled rotors.

__Steps:__

- Create two shafts: Start by defining two rotor shafts using the `ShaftElement` class. Each shaft can have its own properties like material, length, and diameter.

- Join shafts with coupling: Use the `CouplingElement` class to connect the two shafts by inserting the elements in a list following a certain order.

### 2.3.1 Define the shafts with their respective dimensions and properties

In [19]:
# Shaft 1 - Left
L1 = [300, 92, 200, 200, 92, 300]  # Segment lengths (in mm)
r1 = [61.5, 75, 75, 75, 75, 61.5]  # Segment radii (in mm)
shaft1 = [
    rs.ShaftElement(
        L=L1[i] * 1e-3,  # Convert length to meters
        idl=0.0,  # Inner diameter in meters
        odl=r1[i] * 2e-3,  # Outer diameter in meters
        material=steel,  # Material used for the shaft
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(L1))
]

In [20]:
# Shaft 2 - Right
L2 = [80, 200, 200, 640]  # Segment lengths (in mm)
r2 = [160.5, 160.5, 130.5, 130.5]  # Segment radii (in mm)
shaft2 = [
    rs.ShaftElement(
        L=L2[i] * 1e-3,  # Convert length to meters
        idl=0.0,  # Inner diameter in meters
        odl=r2[i] * 2e-3,  # Outer diameter in meters
        material=steel,  # Material used for the shaft
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(L2))
]

### 2.3.2 Define the coupling element that connects the two shafts

In this example, only the torsional stiffness is adopted. However, you can pass any of the following arguments (if not provided, the default value is zero):

- `k_x`, 

- `k_y`,

- `k_z` (axial stiffness), 

- `kr_x`, 

- `kr_y`,

- `kr_z` (torsional stiffness),

considering that the stiffness matrix $\mathbf{K}$ will be assembled like this:

\begin{equation}
\mathbf{K} = 
\begin{bmatrix}
k_x & 0 & 0 & 0 & 0 & 0 & -k_x & 0 & 0 & 0 & 0 & 0 \\
0 & k_y & 0 & 0 & 0 & 0 & 0 & -k_y & 0 & 0 & 0 & 0 \\
0 & 0 & k_z & 0 & 0 & 0 & 0 & 0 & -k_z & 0 & 0 & 0 \\
0 & 0 & 0 & k_{r_x} & 0 & 0 & 0 & 0 & 0 & -k_{r_x} & 0 & 0 \\
0 & 0 & 0 & 0 & k_{r_y} & 0 & 0 & 0 & 0 & 0 & -k_{r_y} & 0 \\
0 & 0 & 0 & 0 & 0 & k_{r_z} & 0 & 0 & 0 & 0 & 0 & -k_{r_z} \\
-k_x & 0 & 0 & 0 & 0 & 0 & k_x & 0 & 0 & 0 & 0 & 0 \\
0 & -k_y & 0 & 0 & 0 & 0 & 0 & k_y & 0 & 0 & 0 & 0 \\
0 & 0 & -k_z & 0 & 0 & 0 & 0 & 0 & k_z & 0 & 0 & 0 \\
0 & 0 & 0 & -k_{r_x} & 0 & 0 & 0 & 0 & 0 & k_{r_x} & 0 & 0 \\
0 & 0 & 0 & 0 & -k_{r_y} & 0 & 0 & 0 & 0 & 0 & k_{r_y} & 0 \\
0 & 0 & 0 & 0 & 0 & -k_{r_z} & 0 & 0 & 0 & 0 & 0 & k_{r_z}
\end{bmatrix}
\end{equation}

>**Note**: Although not demonstrated here, the same logic used for stiffness coefficients can also be applied to damping coefficients (`ct_x`, `ct_y`, `ct_z`, `cr_x`, `cr_y`, `cr_z`).

In [21]:
# Mass at each station (e.g., hub mass of the coupling)
mass_left_station = 151 / 2
mass_right_station = 151 / 2

# Polar moment of inertia of the coupling
Ip = 3.48

# Torsional stiffness
torsional_stiffness = 3.04e6

# Create the coupling element
coupling = rs.CouplingElement(
    m_l=mass_left_station,  # Mass on the left station (in kg)
    m_r=mass_right_station,  # Mass on the right station (in kg)
    Ip_l=Ip
    / 2,  # Polar moment of inertia on the left station (assuming equal distribution in kg·m²)
    Ip_r=Ip
    / 2,  # Polar moment of inertia on the right station (assuming equal distribution in kg·m²)
    kr_z=torsional_stiffness,  # Torsional stiffness (in N·m/rad)
)
coupling

CouplingElement(m=151.0, Ip=3.48, n=None)

### 2.3.3 Coupling all elements

Combine all elements into a single list for the rotor model.
The elements must be in order.

In [22]:
shaft_elements = [*shaft1, coupling, *shaft2]
shaft_elements

[ShaftElement(L=0.3, idl=0.0, idr=0.0, odl=0.123,  odr=0.123, material='Steel', n=None),
 ShaftElement(L=0.092, idl=0.0, idr=0.0, odl=0.15,  odr=0.15, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.0, idr=0.0, odl=0.15,  odr=0.15, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.0, idr=0.0, odl=0.15,  odr=0.15, material='Steel', n=None),
 ShaftElement(L=0.092, idl=0.0, idr=0.0, odl=0.15,  odr=0.15, material='Steel', n=None),
 ShaftElement(L=0.3, idl=0.0, idr=0.0, odl=0.123,  odr=0.123, material='Steel', n=None),
 CouplingElement(m=151.0, Ip=3.48, n=None),
 ShaftElement(L=0.08, idl=0.0, idr=0.0, odl=0.321,  odr=0.321, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.0, idr=0.0, odl=0.321,  odr=0.321, material='Steel', n=None),
 ShaftElement(L=0.2, idl=0.0, idr=0.0, odl=0.261,  odr=0.261, material='Steel', n=None),
 ShaftElement(L=0.64, idl=0.0, idr=0.0, odl=0.261,  odr=0.261, material='Steel', n=None)]

# Section 3: DiskElement Class

The class `DiskElement` allows you to create disk elements, representing rotor equipments which can be considered only to add mass and inertia to the system, disregarding the stiffness.

ROSS offers three ways to create a disk element:

1. Inputting mass and inertia data

2. Inputting geometrical and material data

3. From Excel table

## 3.1 Creating disk elements from inertia properties

If you have access to the mass and inertia properties of a equipment, you can input the data directly to the element.

Disk elements are useful to represent equipments which mass and inertia are significant, but the stiffness can be neglected.

### 3.1.1 Creating a single disk element

This example below shows how to instantiate a disk element according to the mass and inertia properties:

In [23]:
disk = rs.DiskElement(n=0, m=32.58, Ip=0.178, Id=0.329, tag="Disk")
disk

DiskElement(Id=0.329, Ip=0.178, m=32.58, color='Firebrick', n=0, scale_factor=1.0, tag='Disk')

### 3.1.2 Creating a list of disk element

This example below shows how to create a list of disk element according to the mass and inertia properties. The logic is the same applied to shaft elements.

In [24]:
# OPTION No.1:
# Using zip() method
n_list = [2, 4]
m_list = [32.6, 35.8]
Id_list = [0.17808928, 0.17808928]
Ip_list = [0.32956362, 0.38372842]
disk_elements = [
    rs.DiskElement(
        n=n,
        m=m,
        Id=Id,
        Ip=Ip,
    )
    for n, m, Id, Ip in zip(n_list, m_list, Id_list, Ip_list, strict=True)
]
disk_elements

[DiskElement(Id=0.17809, Ip=0.32956, m=32.6, color='Firebrick', n=2, scale_factor=1.0, tag=None),
 DiskElement(Id=0.17809, Ip=0.38373, m=35.8, color='Firebrick', n=4, scale_factor=1.0, tag=None)]

In [25]:
# OPTION No.2:
# Using list index
n_list = [2, 4]
m_list = [32.6, 35.8]
Id_list = [0.17808928, 0.17808928]
Ip_list = [0.32956362, 0.38372842]
N = len(n_list)
disk_elements = [
    rs.DiskElement(
        n=n_list[i],
        m=m_list[i],
        Id=Id_list[i],
        Ip=Ip_list[i],
    )
    for i in range(N)
]
disk_elements

[DiskElement(Id=0.17809, Ip=0.32956, m=32.6, color='Firebrick', n=2, scale_factor=1.0, tag=None),
 DiskElement(Id=0.17809, Ip=0.38373, m=35.8, color='Firebrick', n=4, scale_factor=1.0, tag=None)]

## 3.2 Creating disk elements from geometrical properties

Besides the instantiation previously explained, there is a way to instantiate a `DiskElement` with only geometrical parameters (an approximation for cylindrical disks) and the disk’s material, as we can see in the following code. In this case, the class method `rs.DiskElement.from_geometry()` can be used.

ROSS will take geometrical parameters (outer and inner diameters, and width) and convert them into mass and inertia data. Once again, considering the disk as a cylinder.

### 3.2.1 Creating a single disk element

This example below shows how to instantiate a disk element according to the geometrical and material properties:

In [26]:
disk1 = rs.DiskElement.from_geometry(
    n=4, material=steel, width=0.07, i_d=0.05, o_d=0.28
)
print(disk1)

print("=" * 76)
print(f"Disk mass:              {disk1.m}")
print(f"Disk polar inertia:     {disk1.Ip}")
print(f"Disk diametral inertia: {disk1.Id}")

Tag:                      None
Node:                     4
Mass           (kg):      32.59
Diam. inertia  (kg*m**2): 0.17809
Polar. inertia (kg*m**2): 0.32956
Disk mass:              32.58972765304033
Disk polar inertia:     0.32956362089137037
Disk diametral inertia: 0.17808928257067666


### 3.2.2 Creating a list of disk element

This example below shows how to create a list of disk element according to the geometrical and material properties. The logic is the same applied to shaft elements.

In [27]:
# OPTION No.1:
# Using zip() method
n_list = [2, 4]
width_list = [0.7, 0.7]
i_d_list = [0.05, 0.05]
o_d_list = [0.15, 0.18]
disk_elements = [
    rs.DiskElement.from_geometry(
        n=n,
        material=steel,
        width=width,
        i_d=i_d,
        o_d=o_d,
    )
    for n, width, i_d, o_d in zip(n_list, width_list, i_d_list, o_d_list, strict=True)
]
disk_elements

[DiskElement(Id=3.6408, Ip=0.26836, m=85.875, color='Firebrick', n=2, scale_factor=1.0, tag=None),
 DiskElement(Id=5.5224, Ip=0.56007, m=128.38, color='Firebrick', n=4, scale_factor=1.0, tag=None)]

In [28]:
# OPTION No.2:
# Using list index
n_list = [2, 4]
width_list = [0.7, 0.7]
i_d_list = [0.05, 0.05]
o_d_list = [0.15, 0.18]
N = len(n_list)
disk_elements = [
    rs.DiskElement.from_geometry(
        n=n_list[i],
        material=steel,
        width=width_list[i],
        i_d=i_d_list[i],
        o_d=o_d_list[i],
    )
    for i in range(N)
]
disk_elements

[DiskElement(Id=3.6408, Ip=0.26836, m=85.875, color='Firebrick', n=2, scale_factor=1.0, tag=None),
 DiskElement(Id=5.5224, Ip=0.56007, m=128.38, color='Firebrick', n=4, scale_factor=1.0, tag=None)]

# Section 4: BearingElement and SealElement Classes

**_Bearings_** are passive or active elements that provide stiffness and damping support to a rotating shaft. In a finite-element rotor model, each bearing element connects a rotor node to ground (or to another node) and contributes to the global stiffness and damping matrices through a set of dynamic coefficients.

ROSS organises its bearing classes into three groups:

| Group | Classes | Description |
|-------|---------|-------------|
| **General / parametric** | `BearingElement`, `BallBearingElement`, `RollerBearingElement`, `CylindricalBearing` | Coefficients supplied directly or computed from a simple analytical model |
| **Thermo-Hydro-Dynamic (THD)** | `PlainJournal`, `TiltingPad`, `ThrustPad`, `SqueezeFilmDamper` | Coefficients computed from numerical or analytical solution of the Reynolds equation, with optional thermal coupling |
| **Active Magnetic Bearings (AMB)** | `MagneticBearingElement` | Coefficients derived from electromagnetic parameters and a closed-loop PID or custom transfer-function controller |

All bearing classes inherit from `BearingElement` and can be inserted directly into a `Rotor` assembly.

**_Seals_** represent fluid-induced forces in annular gaps (labyrinths, hole-pattern / honeycomb, hybrids). Like bearings, they add stiffness and damping at a rotor node, but they do **_not_** support the rotor weight and are therefore excluded from static analysis.

When coefficients are already known (from a vendor, a test, or an external tool), use `SealElement` — a `BearingElement` subclass with the handling above. Physics-based models compute leakage and rotordynamic coefficients from geometry and operating conditions:

| Group | Classes | Description |
|-------|---------|-------------|
| **Parametric** | `SealElement` | Coefficients supplied directly (constant or speed-dependent) |
| **Physics-based** | `LabyrinthSeal`, `HolePatternSeal`, `HybridSeal` | Coefficients computed from compressible-flow / bulk-flow models |

Dedicated tutorials for bearings and seals:

- **[General Bearing Classes](tutorial_bearings_part_1.ipynb)** — `BearingElement`, `BallBearingElement`, `RollerBearingElement`, `CylindricalBearing`

- **[Fluid-Film Bearings](tutorial_bearings_part_2.ipynb)** — `PlainJournal`, `TiltingPad`, `ThrustPad`, `SqueezeFilmDamper`

- **[Magnetic Bearings](tutorial_bearings_part_3.ipynb)** — `MagneticBearingElement`

- **[Seals](tutorial_seals.ipynb)** — `SealElement`, `LabyrinthSeal`, `HolePatternSeal`, `HybridSeal`

# Section 5: PointMass Class


The `PointMass` class creates a point mass element. This element can be used to link other elements in the analysis. The mass provided can be different on the x and y direction (e.g. different support inertia for **_x_** and **_y_** directions).

`PointMass` also keeps the mass, stiffness, damping and gyroscopic matrices sizes consistence. When adding 2 bearing elements in series, it opens a new node with new degrees of freedom (DoF) and expands the stiffness and damping matrices. For this reason, it's necessary to add mass values to those DoF to match the matrices sizes.

If you input the argument `m`, the code automatically replicate the mass value for both directions **_x_** and **_y_**.

Let's see an example of creating point masses:

In [29]:
# inputting m
p0 = rs.PointMass(n=0, m=2)
p0.M()  # returns de mass matrices for the element

array([[2., 0., 0.],
       [0., 2., 0.],
       [0., 0., 2.]])

In [30]:
# inputting mx and my
p1 = rs.PointMass(n=0, mx=2, my=3)
p1.M()

array([[2., 0., 0.],
       [0., 3., 0.],
       [0., 0., 0.]])

Units with Pint are covered in [Tutorial - Units](tutorial_units.ipynb).